# Кого увековечивает топонимика города? Анализ названий улиц Новосибирска

**Digital Humanities · финальный проект**

Эта тетрадка – воспроизводимый анализ итогового датасета `data/novosibirsk_streets.json`. Сбор и обогащение данных описаны в отдельной тетрадке `../data_collector.ipynb`.

## 1. Исследовательский вопрос

**Кого увековечивает городская топонимика Новосибирска?** Город понимается как «текст», а названия улиц – как идеологические маркеры исторической памяти. Проверяемые подвопросы:

1. **Гендер**: какова доля улиц, названных в честь женщин?
2. **Профессия**: какие сферы деятельности доминируют?
3. **Эпоха**: к каким историческим периодам принадлежат увековеченные?
4. **Связки**: складывается ли из срезов «сфера × эпоха» связная биография города?

## 2. Данные и подготовка

Загружаем итоговый датасет и смотрим его структуру.

In [ ]:
import json, re, os
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 110

ACCENT, MALE, FEMALE, NEUTRAL = '#7e44ff', '#3b86db', '#e0699f', '#8a8fa3'

with open('../data/novosibirsk_streets.json', encoding='utf-8') as f:
    df = pd.DataFrame(json.load(f))
named = df[df['named_after_person'] == True].copy()
print('всего записей:', len(df), '| именных:', len(named))
df[['street','person','gender','occupation','epoch','named_after_person']].head()

### Структура полей

Ключевые поля: `street` (название), `person` (персона), `gender`, `occupation` (сфера), `epoch`, `birth_year`/`death_year`, `lat`/`lon` (координаты), `wikidata_url`/`wikipedia_url` (источники), `match_source` (как установлена атрибуция).

## 3. Масштаб корпуса

In [ ]:
total = len(df); n = len(named)
print(f'Всего топонимов: {total}')
print(f'Именных (в честь человека): {n} ({n/total*100:.1f}%)')
print(f'Уникальных персон: {named["person"].nunique()}')
print(f'Со ссылкой Wikipedia: {named["wikipedia_url"].notna().sum()} | Wikidata: {named["wikidata_url"].notna().sum()}')

> Только **−19%** городских топонимов названы в честь людей. Важная оговорка: далее счёт идёт по *топонимам*, а не по *уникальным людям* – одна фигура может быть увековечена несколько раз.

## 4. Гендерный дисбаланс

![Гендер](figures/01_gender.png)

In [ ]:
gender = named['gender'].fillna('Не определён').value_counts()
order = [g for g in ['Мужской','Женский','Множественное','Не определён'] if g in gender.index]
vals = [gender[g] for g in order]
cmap = {'Мужской':MALE,'Женский':FEMALE,'Множественное':NEUTRAL,'Не определён':NEUTRAL}
fig, ax = plt.subplots(figsize=(6,4))
bars = ax.bar(order, vals, color=[cmap[g] for g in order])
for b,v in zip(bars, vals):
    ax.text(b.get_x()+b.get_width()/2, v+3, f'{v}\n{v/len(named)*100:.1f}%', ha='center', va='bottom')
ax.set_title('Гендерный состав именных улиц'); ax.set_ylabel('Топонимы')
ax.spines[['top','right']].set_visible(False); plt.tight_layout(); plt.show()

**Интерпретация.** Доля «женских» улиц – около **5%**, у нижней границы общемирового паттерна (5–10%). Реальный дисбаланс ещё глубже из-за повторов (см. раздел 9).

## 5. Профессиональный срез: кто «герои города»

![Профессии](figures/02_occupation.png)

In [ ]:
occ = named['occupation'].fillna('Другое').value_counts().sort_values()
fig, ax = plt.subplots(figsize=(8,5))
ax.barh(occ.index, occ.values, color=ACCENT)
for i,v in enumerate(occ.values):
    ax.text(v+1, i, f'{v} ({v/len(named)*100:.0f}%)', va='center', fontsize=9)
ax.set_title('Сферы деятельности увековеченных'); ax.set_xlabel('Топонимы')
ax.spines[['top','right']].set_visible(False); plt.tight_layout(); plt.show()

**Интерпретация.** Четыре сферы – **военное дело, литература, наука, политика** – дают около **73%** именных улиц. Это классический советский пантеон: воин-герой, писатель-классик, учёный, революционер.

## 6. Хронотоп памяти: эпохи

![Эпохи](figures/03_epoch.png)

In [ ]:
ROMAN = {'I':1,'V':5,'X':10,'L':50,'C':100,'D':500,'M':1000}
def roman_to_int(s):
    s=s.strip().upper(); t=0; prev=0
    for ch in reversed(s):
        if ch not in ROMAN: return None
        v=ROMAN[ch]; t += -v if v<prev else v; prev=max(prev,v)
    return t
def epoch_key(label):
    if not label: return 9999
    if 'Древн' in label: return -1
    m = re.match(r'\s*([IVXLCDM]+)', str(label))
    return roman_to_int(m.group(1)) if m else 9998
epoch = named['epoch'].fillna('Неизвестно').value_counts()
epoch = epoch.reindex(sorted(epoch.index, key=epoch_key))
fig, ax = plt.subplots(figsize=(9,4.5))
ax.bar(epoch.index, epoch.values, color=ACCENT)
for i,v in enumerate(epoch.values): ax.text(i, v+2, str(v), ha='center', fontsize=8)
ax.set_title('Эпохи увековеченных личностей'); ax.set_ylabel('Топонимы')
plt.xticks(rotation=45, ha='right', fontsize=8)
ax.spines[['top','right']].set_visible(False); plt.tight_layout(); plt.show()

**Интерпретация.** Почти **87%** памяти сосредоточено на XIX–XX веках. Это ожидаемо для молодого города (Новосибирск основан в 1893): уличная сеть формировалась уже в советскую эпоху.

## 7. Гендер × сфера: где «разрешено» быть женщиной

![Гендер × сфера](figures/04_gender_occupation.png)

In [ ]:
ct = pd.crosstab(named['occupation'].fillna('Другое'), named['gender'].fillna('Не опр.'))
for c in ['Мужской','Женский']:
    if c not in ct.columns: ct[c]=0
ct['всего']=ct.sum(axis=1); ct['женщины %']=(ct['Женский']/ct['всего']*100).round(1)
ct = ct.sort_values('всего')
fig, ax = plt.subplots(figsize=(8,5))
ax.barh(ct.index, ct['Мужской'], color=MALE, label='Мужчины')
ax.barh(ct.index, ct['Женский'], left=ct['Мужской'], color=FEMALE, label='Женщины')
ax.set_title('Гендер по сферам'); ax.set_xlabel('Топонимы'); ax.legend()
ax.spines[['top','right']].set_visible(False); plt.tight_layout(); plt.show()
ct[['Мужской','Женский','всего','женщины %']].sort_values('всего', ascending=False)

**Интерпретация.** Женские имена концентрируются в военной сфере (героини/жертвы войны). В промышленности, транспорте, спорте, медицине, географии – **ни одной женщины**.

## 8. Война как ядро памяти (годы смерти)

![Годы смерти](figures/05_death_years.png)

In [ ]:
def year(v):
    m = re.search(r'(\d{3,4})', str(v)) if v is not None else None
    return int(m.group(1)) if m else None
named['death_y'] = named['death_year'].map(year)
dy = named['death_y'].dropna(); dy = dy[(dy>=1700)&(dy<=2025)]
war = named[(named['death_y']>=1941)&(named['death_y']<=1945)]
mil = named[named['occupation']=='Военное дело']
print(f'Погибли в 1941–1945: {len(war)} | из них военных: {((mil["death_y"]>=1941)&(mil["death_y"]<=1945)).sum()} из {len(mil)}')
fig, ax = plt.subplots(figsize=(9,4))
ax.hist(dy, bins=range(1700,2030,10), color=NEUTRAL, edgecolor='white')
ax.axvspan(1941,1945, color='#e05a5a', alpha=0.35, label='1941–1945')
ax.set_title('Годы смерти увековеченных'); ax.set_xlabel('Год'); ax.set_ylabel('Персоны'); ax.legend()
ax.spines[['top','right']].set_visible(False); plt.tight_layout(); plt.show()

**Интерпретация.** Ровно половина военных (62 из 124) погибли в 1941–1945. Великая Отечественная война – единственное событие, давшее столь плотный пласт памяти.

## 9. «Биография города»: сфера × эпоха

![Сфера × эпоха](figures/06_occupation_epoch.png)

In [ ]:
named['epoch_f'] = named['epoch'].fillna('Неизвестно')
occ_order = named['occupation'].fillna('Другое').value_counts().index.tolist()
ep_order = sorted(named['epoch_f'].unique(), key=epoch_key)
heat = pd.crosstab(named['occupation'].fillna('Другое'), named['epoch_f']).reindex(index=occ_order, columns=ep_order).fillna(0)
fig, ax = plt.subplots(figsize=(11,6))
im = ax.imshow(heat.values, aspect='auto', cmap='BuPu')
ax.set_xticks(range(len(ep_order))); ax.set_xticklabels(ep_order, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(occ_order))); ax.set_yticklabels(occ_order, fontsize=9)
for i in range(len(occ_order)):
    for j in range(len(ep_order)):
        val=int(heat.values[i,j])
        if val: ax.text(j,i,val,ha='center',va='center',fontsize=7,color='#222' if val<heat.values.max()*0.6 else 'white')
ax.set_title('Сфера деятельности × эпоха'); plt.colorbar(im, ax=ax, label='Топонимы')
plt.tight_layout(); plt.show()

**Интерпретация.** Срезы складываются в биографию советского города: военные и политики – XX век (война и революция), наука – две волны (имперская классика + поколение Академгородка), литература и искусство – дореволюционный общероссийский канон.

## 10. Повторы: топонимы ≠ люди

In [ ]:
rep = named['person'].value_counts(); rep = rep[rep>1]
print(f'Персон, увековеченных >1 раза: {len(rep)}')
rep.head(10)

Повторное имянаречение усиливает присутствие фигуры в городском тексте – это само по себе маркер памяти.

## 11. Сетевой взгляд (опционально, networkx)

Двудольный граф «сфера – эпоха»: рёбро соединяет сферу и эпоху, если есть персоны на их пересечении. Полноценный интерактивный граф персон реализован в дашборде (`dashboard/`).

In [ ]:
try:
    import networkx as nx
    g = nx.Graph()
    sub = named.dropna(subset=['occupation'])
    for occ_, ep_ in zip(sub['occupation'], sub['epoch_f']):
        g.add_edge(f'сфера: {occ_}', f'эпоха: {ep_}', weight=g.get_edge_data(f'сфера: {occ_}', f'эпоха: {ep_}', {}).get('weight',0)+1)
    print('узлов:', g.number_of_nodes(), '| рёбер:', g.number_of_edges())
    fig, ax = plt.subplots(figsize=(11,8))
    pos = nx.spring_layout(g, k=0.7, seed=42)
    weights = [g[u][v]['weight'] for u,v in g.edges()]
    nx.draw_networkx_edges(g, pos, width=[0.3+w*0.06 for w in weights], edge_color='#bbbccd', ax=ax)
    occ_nodes = [n for n in g.nodes if n.startswith('сфера')]
    ep_nodes = [n for n in g.nodes if n.startswith('эпоха')]
    nx.draw_networkx_nodes(g, pos, nodelist=occ_nodes, node_color=ACCENT, node_size=700, ax=ax)
    nx.draw_networkx_nodes(g, pos, nodelist=ep_nodes, node_color=MALE, node_size=400, ax=ax)
    nx.draw_networkx_labels(g, pos, font_size=7, ax=ax)
    ax.set_title('Граф «сфера – эпоха»'); ax.axis('off'); plt.tight_layout(); plt.show()
except ImportError:
    print('Для графа установите networkx:  pip install networkx')

## 12. Выводы, ограничения и перспективы

**Ответ на вопрос.** Городской текст Новосибирска – маскулинный (≈95% мужчин), милитаризированный (военное дело – крупнейшая сфера) и хронологически сжатый до XIX–XX веков. Это портрет советского индустриально-научного города. Подробные выводы – в `../ANALYTICAL_SUMMARY.md` и `../RESEARCH.md`.

**Ограничения.** (1) Один город – нет межгородского сравнения. (2) Счёт по топонимам, не по людям. (3) Граница «именности» условна; часть атрибуций – ручная. (4) Охват именных улиц (~19%) неполон.

**Перспективы.** Сравнение с Москвой/СПб; анализ по уникальным личностям; пространственная кластеризация («военные» и «литературные» районы); доверификация недостающих имён.